# Train the Naruto seal detector (GPU)

Your laptop is CPU-only, so train here on Colab's free GPU, then download `best.pt`.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then fill in your
Roboflow API key in the download cell and Runtime → **Run all**.

In [ ]:
# 1. Confirm a GPU is attached (should print a table, not an error).
!nvidia-smi

In [ ]:
# 2. Install dependencies.
%pip install -q ultralytics roboflow pyyaml

In [ ]:
# 3. Download the dataset. Paste your Roboflow API key below.
# Change workspace/project/version if you picked a different dataset on Roboflow Universe.
from roboflow import Roboflow

rf = Roboflow(api_key="PASTE_YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("vgu-aeaes").project("naruto-hand-sign")
dataset = project.version(1).download("yolov8")
print("Downloaded to:", dataset.location)

In [ ]:
# 4. Look at the class names. WRITE DOWN the one that is the Kage Bunshin clone/cross
#    seal (or whichever seal you can reliably make) — you'll put it in jutsu/config.py.
import yaml

cfg = yaml.safe_load(open(f"{dataset.location}/data.yaml"))
print("Classes:", cfg["names"])

In [ ]:
# 5. Train. ~20-40 min on a T4 for 50 epochs. Raise epochs for better accuracy.
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(data=f"{dataset.location}/data.yaml", epochs=50, imgsz=640)
print("Weights saved in:", model.trainer.save_dir)

In [ ]:
# 6. Download the trained weights to your computer.
from google.colab import files

files.download(f"{model.trainer.save_dir}/weights/best.pt")

## Back on your laptop

1. Move the downloaded `best.pt` to `cv/models/seal_detector.pt`.
2. In `jutsu/config.py` set `TRIGGER_SEAL` to the exact class name from cell 4.
3. `python app.py` — form the seal in front of the webcam to cast.